## Architecture Overview

This notebook deploys using the following architecture:

```
GitHub Repository (manuka-dsit-mcp-demo)
├─ custom_mcps/dft-mcp-server/  ← App source code
│
└─→ Databricks Workspace
    │
    ├─ Repos API: Sync to /Workspace/Repos/
    │
    ├─ Workspace API: Upload files to app directory
    │  └─ /Workspace/Users/<user>/dft-mcp-server-app/
    │
    └─ Apps API: Create & deploy
       └─ Databricks Apps (HTTP server running)
           └─ Available at: https://workspace.cloud.databricks.com/apps/<app-url>
```

### What Each Cell Does

1. **Cell 1** - Configuration
   - Sets up repo paths, workspace paths, app names
   - Customizable for different apps or workspace locations

2. **Cell 2** - SDK Setup
   - Upgrades Databricks SDK to ≥0.88.0
   - Restarts Python kernel to load new SDK version

3. **Cell 3** - Deployment
   - Uses Repos API to sync GitHub repo to workspace
   - Uses Workspace API to upload files with parent directory creation
   - Uses Apps API to create app (if new) or update existing
   - Deploys and waits for completion


# Deploy DfT MCP Server via Databricks SDK

This notebook deploys the `dft-mcp-server` to Databricks Apps using the Python SDK and Databricks Repos API for source synchronization.

## Overview

- Syncs your MCP server repository to Databricks Workspace Repos
- Uploads source files to the workspace
- Creates and deploys the app using the Databricks SDK
- Provides real-time deployment progress tracking

## Prerequisites
- Running inside a Databricks notebook/cluster
- Databricks workspace scoped to the current user
- Write access to workspace folders
- GitHub repo with MCP server code (or local repo sync)

## Deployment Methods

**This notebook (SDK approach)**:
- Interactive notebook-based deployment
- Good for development and testing
- Real-time feedback and customization
- Requires Python SDK >=0.88.0

**Alternative: CLI Asset Bundle approach**:
- One-command deployment: `bash scripts/deploy_bundle.sh dev`
- From repo root on local machine or workspace terminal
- Better for CI/CD and production deployments
- Uses `databricks.yml` configuration

## Run Order

1. **Cell 1**: Configuration (customize paths and app name)
2. **Cell 2**: SDK upgrade and environment setup
3. **Cell 3**: Deploy (sync repo → upload files → create/deploy app)

In [0]:
import os

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat

# Get current username
w = WorkspaceClient()
current_username = w.current_user.me().user_name
w.workspace.mkdirs(f"/Workspace/Users/{current_username}/mcp_server")
# Repo settings 
REPO_URL = "https://github.com/LeoXLiu-DS/manuka-dsit-mcp-demo.git"
REPO_PROVIDER = "gitHub"
REPO_PATH = f"/Workspace/Users/{current_username}/mcp_server/manuka-dsit-mcp-demo"

# App deploy settings
REPO_DIR = f"{REPO_PATH}/custom_mcps/dft-mcp-server"
WS_PATH = "/Workspace/Users/{current_username}/mcp_server/dft-mcp-server-app"
APP_NAME = "mcp-dft-server"

In [0]:
import os
import sys
import subprocess

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat

def run(cmd, cwd=None, check=True):
    print("+", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, check=check)

# Upgrade SDK to get App manifest support
run([sys.executable, "-m", "pip", "install", "--upgrade", "databricks-sdk>=0.88.0"])

# Restart Python to load the new SDK version
dbutils.library.restartPython()

+ /local_disk0/.ephemeral_nfs/envs/pythonEnv-91128a95-afed-4eab-837c-440e453be184/bin/python -m pip install --upgrade databricks-sdk>=0.88.0



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [0]:
import os
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat
from databricks.sdk.service.apps import App, AppDeployment, AppDeploymentMode

# Get current username
w = WorkspaceClient()
current_username = w.current_user.me().user_name

# Repo settings 
REPO_URL = "https://github.com/LeoXLiu-DS/manuka-dsit-mcp-demo.git"
REPO_PROVIDER = "gitHub"
REPO_PATH = f"/Workspace/Users/{current_username}/mcp_server/manuka-dsit-mcp-demo"

# App deploy settings
REPO_DIR = f"{REPO_PATH}/custom_mcps/dft-mcp-server"
WS_PATH = f"/Workspace/Users/{current_username}/mcp_server/dft-mcp-server-app"
APP_NAME = "dft-mcp-server-01"

# Sync repo into Databricks Repos
repos = list(w.repos.list())
repo = next((r for r in repos if r.path == REPO_PATH), None)
if repo:
    repo_id = getattr(repo, "id", getattr(repo, "repo_id", None))
    if repo_id is None:
        raise ValueError("Repo id not found in Repos response")
    w.repos.update(repo_id, branch="main")
else:
    w.repos.create(REPO_URL, provider=REPO_PROVIDER, path=REPO_PATH)

if not os.path.isdir(REPO_DIR):
    raise FileNotFoundError(f"Repo path not found: {REPO_DIR}")

# Upload source tree to workspace
w.workspace.mkdirs(WS_PATH)
created_dirs = set()
for root, _, files in os.walk(REPO_DIR):
    for filename in files:
        local_path = os.path.join(root, filename)
        rel_path = os.path.relpath(local_path, REPO_DIR)
        ws_file_path = f"{WS_PATH}/{rel_path}"
        ws_parent = os.path.dirname(ws_file_path)
        if ws_parent not in created_dirs:
            w.workspace.mkdirs(ws_parent)
            created_dirs.add(ws_parent)
        with open(local_path, "rb") as handle:
            w.workspace.upload(
                ws_file_path,
                handle.read(),
                format=ImportFormat.AUTO,
                overwrite=True,
            )

# Create app if it doesn't exist
try:
    existing_app = w.apps.get(APP_NAME)
    print(f"App {APP_NAME} already exists")
    
    # Wait for any active deployment to complete
    if existing_app.pending_deployment:
        print("Waiting for active deployment to complete...")
        max_wait = 600
        waited = 0
        while waited < max_wait:
            app_status = w.apps.get(APP_NAME)
            if not app_status.pending_deployment:
                print("Active deployment completed")
                break
            time.sleep(10)
            waited += 10
        else:
            raise TimeoutError(f"Deployment still in progress after {max_wait} seconds")
except Exception as e:
    if "does not exist" in str(e).lower() or "not found" in str(e).lower():
        app = App(name=APP_NAME)
        w.apps.create_and_wait(app=app)
        print(f"App {APP_NAME} created")
    else:
        raise

# Deploy the app
deployment = AppDeployment(
    source_code_path=WS_PATH,
    mode=AppDeploymentMode.SNAPSHOT
)
result = w.apps.deploy_and_wait(app_name=APP_NAME, app_deployment=deployment)
print(f"App deployed: {result.status}")

App dft-mcp-server-01 already exists
App deployed: AppDeploymentStatus(message='App started successfully', state=<AppDeploymentState.SUCCEEDED: 'SUCCEEDED'>)
